# LLM-FFT Analysis Notebook

Replicates all analyses and figures from:
> **Addressing the alignment problem in transportation policy making: an LLM approach**

## Sections
1. Setup & Configuration
2. Figure 4 — Policy Space Utility Scatter Plots
3. Helper Functions — Loading Simulation Results
4. Figure 5 — Vote Count Distribution (single round)
5. Table 2 — Single-round statistics (winner, mean policy values, entropy)
6. Table 3 — Multi-round summary (winner counts, entropy, lever entropy)
7. Figure 6/7 — 3D Policy Space Bubble Charts
8. Figure 8 — VADER Sentiment Analysis
9. Section 4.4 — OLS Regression
10. Section 4.5 — Houston vs Chicago Comparison

## 1. Setup & Configuration

In [ ]:
import os
import json
import ast
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.stats import entropy as scipy_entropy
from collections import Counter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FIG_DIR = Path('./figures')
FIG_DIR.mkdir(exist_ok=True)
print('Setup complete.')

In [ ]:
# ── Experiment configuration ──────────────────────────────────────────────────
# Directory convention from simulate.py:
#   results/{city}_{model_short}_{mode}[_info]
#   model_short = model_name.split('-')[0]  →  'gpt' or 'claude'
#
# Edit these paths to match where you stored your simulation outputs.

RESULTS_ROOT = Path('./results')

CHI_EXPERIMENTS = {
    'CHI-com GPT-4o (ranked)'      : 'chicago_gpt_ranked',
    'CHI-know GPT-4o (ranked)'     : 'chicago_gpt_ranked_info',
    'CHI-avg GPT-4o'               : 'chicago_gpt_average',
    'CHI-com Claude (ranked)'      : 'chicago_claude_ranked',
    'CHI-know Claude (ranked)'     : 'chicago_claude_ranked_info',
    'CHI-com GPT-4o (approval-5)'  : 'chicago_gpt_approval',
    'CHI-com GPT-4o (approval-all)': 'chicago_gpt_approval-all',
    'CHI-com Claude (approval-5)'  : 'chicago_claude_approval',
    'CHI-com Claude (approval-all)': 'chicago_claude_approval-all',
}

HOU_EXPERIMENTS = {
    'HOU-com GPT-4o (ranked)'      : 'houston_gpt_ranked',
    'HOU-com Claude (ranked)'      : 'houston_claude_ranked',
    'HOU-com GPT-4o (approval-all)': 'houston_gpt_approval-all',
    'HOU-com Claude (approval-all)': 'houston_claude_approval-all',
}

POLICY_UTIL_CHI = Path('./data/policy_simulation_results_util_chicago.csv')
POLICY_UTIL_HOU = Path('./data/policy_simulation_results_util_houston.csv')
POLICY_CSV_CHI  = Path('./data/policy_chicago.csv')
POLICY_CSV_HOU  = Path('./data/policy_houston.csv')
CA_INFO_DIR     = Path('./data/CA_info')
NUM_ROUNDS = 10

print('Configuration loaded.')

## 2. Figure 4 — Policy Space Utility Scatter Plots

Scatter plots of $U_k$ vs $u_k$, $\gamma_k$, $G_k$ for all 27 policies.  
Color = tax level, marker shape = fare level, fill = driver fee level.

In [ ]:
def plot_policy_scatter(
    util_csv, x_col='Utotal', y_col='Umin',
    x_label=r'$U_k$', y_label=r'$u_k$',
    figsize=(6, 4), legend_loc=(0.01, 0.99), save_path=None,
):
    """
    Scatter plot of two utility metrics across all 27 policies.
    Encoding: color=tax, marker=fare, fill=driver_fee.
    """
    df = pd.read_csv(util_csv)
    tax_levels  = sorted(df['tax_percentage'].unique())
    fare_levels = sorted(df['fare'].unique())
    fee_levels  = sorted(df['driver_fee'].unique())

    color_map  = {t: c for t, c in zip(tax_levels,  sns.color_palette('tab10', 3))}
    marker_map = {f: m for f, m in zip(fare_levels, ['o', 's', '^'])}
    fill_map   = {fee_levels[0]: 'none', fee_levels[1]: 'half', fee_levels[2]: 'full'}

    fig, ax = plt.subplots(figsize=figsize)
    for _, row in df.iterrows():
        c, mk = color_map[row['tax_percentage']], marker_map[row['fare']]
        fst = fill_map[row['driver_fee']]
        kw = dict(color=c, marker=mk, s=90, linewidth=1.2)
        if fst == 'none':
            ax.scatter(row[x_col], row[y_col], facecolors='none', edgecolors=c, **kw)
        elif fst == 'half':
            ax.scatter(row[x_col], row[y_col], facecolors=c, edgecolors=c, alpha=0.45, s=110, **{k:v for k,v in kw.items() if k!='s'})
        else:
            ax.scatter(row[x_col], row[y_col], facecolors=c, edgecolors=c, **kw)

    tax_h  = [mpatches.Patch(facecolor=color_map[t], label=f'Tax {t}%') for t in tax_levels]
    fare_h = [Line2D([0],[0], marker=marker_map[f], color='k', linestyle='', label=f'Fare ${f}', markersize=9) for f in fare_levels]
    fee_h  = [
        mpatches.Patch(facecolor='white', edgecolor='k', label=f'Fee ${fee_levels[0]}'),
        mpatches.Patch(facecolor='grey', edgecolor='k', alpha=0.5, label=f'Fee ${fee_levels[1]}'),
        mpatches.Patch(facecolor='grey', edgecolor='k', label=f'Fee ${fee_levels[2]}'),
    ]
    ax.legend(handles=tax_h+fare_h+fee_h, bbox_to_anchor=legend_loc, loc='upper left', fontsize=9)
    ax.set_xlabel(x_label, fontsize=13); ax.set_ylabel(y_label, fontsize=13)
    ax.tick_params(labelsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()


print('plot_policy_scatter defined.')

In [ ]:
# Figure 4 — Chicago
for y_col, y_label, fn in [
    ('Umin',     r'$u_k$',      'uk'),
    ('Transit%', r'$\gamma_k$', 'gamma'),
    ('Gini',     r'$G_k$',      'Gk'),
]:
    plot_policy_scatter(
        POLICY_UTIL_CHI, x_col='Utotal', y_col=y_col,
        x_label=r'$U_k$', y_label=y_label,
        save_path=FIG_DIR / f'fig4_chicago_Uk_{fn}.png',
    )

In [ ]:
# Figure 4 — Houston
for y_col, y_label, fn in [
    ('Umin',     r'$u_k$',      'uk'),
    ('Transit%', r'$\gamma_k$', 'gamma'),
    ('Gini',     r'$G_k$',      'Gk'),
]:
    plot_policy_scatter(
        POLICY_UTIL_HOU, x_col='Utotal', y_col=y_col,
        x_label=r'$U_k$', y_label=y_label,
        save_path=FIG_DIR / f'fig4_houston_Uk_{fn}.png',
    )

## 3. Helper Functions — Loading Simulation Results

In [ ]:
def load_policy_params(policy_csv: Path) -> pd.DataFrame:
    """Load policy CSV indexed by policy number."""
    df = pd.read_csv(policy_csv)
    return df.set_index('index') if 'index' in df.columns else df


def parse_vote_list(v):
    """Parse a vote-column entry (list or stringified list)."""
    if isinstance(v, list): return v
    try: return ast.literal_eval(str(v))
    except: return []


def compute_policy_entropy(rank1_votes: pd.Series) -> float:
    """Eq. 8: Shannon entropy of rank-1 policy distribution (base 2)."""
    counts = rank1_votes.astype(int).value_counts()
    return float(scipy_entropy(counts / counts.sum(), base=2))


def compute_lever_entropy(rank1_votes: pd.Series, policy_df: pd.DataFrame, lever: str) -> float:
    """Eq. 9: Shannon entropy of lever-level distribution in rank-1 votes."""
    vals = rank1_votes.astype(int).map(policy_df[lever])
    counts = vals.value_counts()
    return float(scipy_entropy(counts / counts.sum(), base=2))


def get_rank1(df: pd.DataFrame, mode: str) -> pd.Series:
    """Extract rank-1 (first-choice) policy IDs from a round DataFrame."""
    if mode == 'ranked' and 'rank1' in df.columns:
        return df['rank1'].dropna().astype(int)
    elif 'vote' in df.columns:
        return df['vote'].apply(lambda v: parse_vote_list(v)[0] if parse_vote_list(v) else np.nan).dropna().astype(int)
    return pd.Series(dtype=int)


def load_all_rank1(exp_dir: Path, mode: str, num_rounds: int = NUM_ROUNDS) -> pd.Series:
    """Collect all rank-1 votes across all rounds."""
    all_votes = []
    for i in range(1, num_rounds + 1):
        csv_path = exp_dir / f'round_{i}' / 'combined_voting_results.csv'
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            all_votes.extend(get_rank1(df, mode).tolist())
    return pd.Series(all_votes, dtype=int)


print('Helper functions defined.')

## 4. Figure 5 — Vote Count Distribution (Single Round)

Bar chart of votes per policy in one round. Red bar = IRV/plurality winner.

In [ ]:
def plot_vote_counts(
    exp_dir: Path, round_num: int = 1, mode: str = 'ranked',
    num_policies: int = 27, title: str = '', save_path=None,
):
    csv_path = exp_dir / f'round_{round_num}' / 'combined_voting_results.csv'
    if not csv_path.exists():
        print(f'File not found: {csv_path}'); return

    df = pd.read_csv(csv_path)
    if mode == 'ranked':
        counts = df['rank1'].astype(int).value_counts().reindex(range(num_policies), fill_value=0)
        ylabel = 'First-choice votes'
    else:
        all_v = []
        for v in df['vote']: all_v.extend(parse_vote_list(v))
        counts = pd.Series(Counter(all_v)).reindex(range(num_policies), fill_value=0)
        ylabel = 'Approval votes'

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')

    irv_path = exp_dir / f'round_{round_num}' / 'irv_summary.json'
    winner = None
    if irv_path.exists():
        winner = json.loads(irv_path.read_text()).get('winning_policy')
        if winner is not None and winner < len(bars):
            bars[winner].set_color('tomato')

    ax.set_title(f'{title}  (Winner: Policy {winner})' if winner is not None else title, fontsize=12)
    ax.set_xlabel('Policy index', fontsize=12); ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xticks(range(num_policies)); ax.tick_params(labelsize=9)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Figure 5 — Compare three voting modes (GPT-4o, Chicago, Round 1)
modes_info = [
    ('CHI-com GPT-4o (ranked)',       'ranked',       'CHI-com GPT-4o\nRanked (rank-1)'),
    ('CHI-com GPT-4o (approval-5)',   'approval',     'CHI-com GPT-4o\n5-Approval'),
    ('CHI-com GPT-4o (approval-all)', 'approval-all', 'CHI-com GPT-4o\nApproval-All'),
]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, (label, mode, subplot_title) in zip(axes, modes_info):
    sub_dir = RESULTS_ROOT / CHI_EXPERIMENTS.get(label, '')
    if not sub_dir.exists():
        ax.text(0.5, 0.5, 'No data\n(run simulation first)', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(subplot_title); continue

    csv_path = sub_dir / 'round_1' / 'combined_voting_results.csv'
    if not csv_path.exists():
        ax.text(0.5, 0.5, 'Missing CSV', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(subplot_title); continue

    df = pd.read_csv(csv_path)
    if mode == 'ranked':
        counts = df['rank1'].astype(int).value_counts().reindex(range(27), fill_value=0)
    else:
        all_v = []
        for v in df['vote']: all_v.extend(parse_vote_list(v))
        counts = pd.Series(Counter(all_v)).reindex(range(27), fill_value=0)

    bars = ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    irv_path = sub_dir / 'round_1' / 'irv_summary.json'
    if irv_path.exists():
        w = json.loads(irv_path.read_text()).get('winning_policy')
        if w is not None and w < len(bars): bars[w].set_color('tomato')
    ax.set_title(subplot_title, fontsize=11)
    ax.set_xlabel('Policy index', fontsize=10); ax.set_ylabel('Votes', fontsize=10)
    ax.set_xticks(range(0, 27, 3))

plt.suptitle('Figure 5 — Vote distribution by voting mode (Round 1)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_vote_counts_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Table 2 — Single-round Statistics

For each experiment and round:
- Winning policy $k^*$, lever values $t^*$, $r^*$, $\tau^*$
- Mean rank-1 lever values: $\bar{t}$, $\bar{r}$, $\bar{\tau}$
- Entropy $E$ (Eq. 8), lever entropy $e_t$, $e_r$, $e_\tau$ (Eq. 9)

In [ ]:
def single_round_stats(exp_dir: Path, round_num: int, policy_df: pd.DataFrame, mode: str = 'ranked') -> dict:
    csv_path = exp_dir / f'round_{round_num}' / 'combined_voting_results.csv'
    irv_path = exp_dir / f'round_{round_num}' / 'irv_summary.json'
    if not csv_path.exists(): return {}

    df = pd.read_csv(csv_path)
    rank1 = get_rank1(df, mode)
    if rank1.empty: return {}

    r1_params = policy_df.loc[rank1]
    result = {
        'round':      round_num,
        'winner':     json.loads(irv_path.read_text()).get('winning_policy') if irv_path.exists() else None,
        'mean_tax':   r1_params['tax_percentage'].mean(),
        'mean_fare':  r1_params['fare'].mean(),
        'mean_fee':   r1_params['driver_fee'].mean(),
        'entropy_E':  compute_policy_entropy(rank1),
        'e_tax':      compute_lever_entropy(rank1, policy_df, 'tax_percentage'),
        'e_fare':     compute_lever_entropy(rank1, policy_df, 'fare'),
        'e_fee':      compute_lever_entropy(rank1, policy_df, 'driver_fee'),
    }
    if result['winner'] is not None and result['winner'] in policy_df.index:
        wp = policy_df.loc[result['winner']]
        result.update({'win_tax': wp['tax_percentage'], 'win_fare': wp['fare'], 'win_fee': wp['driver_fee']})
    return result

In [ ]:
# Table 2 — Chicago, Round 1
policy_df_chi = load_policy_params(POLICY_CSV_CHI)
table2_rows = []

for label, subdir in CHI_EXPERIMENTS.items():
    exp_dir = RESULTS_ROOT / subdir
    if not exp_dir.exists(): continue
    mode = 'approval' if 'approval' in subdir else 'ranked'
    s = single_round_stats(exp_dir, 1, policy_df_chi, mode)
    if s: s['experiment'] = label; table2_rows.append(s)

if table2_rows:
    t2 = pd.DataFrame(table2_rows)
    cols_present = [c for c in ['experiment','winner','win_tax','win_fare','win_fee',
                                 'mean_tax','mean_fare','mean_fee','entropy_E','e_tax','e_fare','e_fee']
                    if c in t2.columns]
    t2 = t2[cols_present].rename(columns={
        'winner':'Winner', 'win_tax':'t*', 'win_fare':'r*', 'win_fee':'τ*',
        'mean_tax':'t̄', 'mean_fare':'r̄', 'mean_fee':'τ̄', 'entropy_E':'E',
        'e_tax':'e_t', 'e_fare':'e_r', 'e_fee':'e_τ',
    })
    pd.set_option('display.float_format', '{:.3f}'.format)
    print('Table 2 — Single-round statistics (Round 1, Chicago):')
    display(t2)
else:
    print('No data — run simulations first.')

## 6. Table 3 — Multi-round Summary

Aggregate across 10 rounds:
- Most-frequent winner and win count
- Mean entropy $\bar{E}$
- Mean lever values with lever entropy: $\bar{t}$ ($e_t$), $\bar{r}$ ($e_r$), $\bar{\tau}$ ($e_\tau$)

In [ ]:
def multi_round_summary(exp_dir: Path, policy_df: pd.DataFrame, mode: str = 'ranked', num_rounds: int = NUM_ROUNDS) -> dict:
    round_stats = [single_round_stats(exp_dir, i, policy_df, mode) for i in range(1, num_rounds+1)]
    round_stats = [s for s in round_stats if s]
    if not round_stats: return {}

    sdf = pd.DataFrame(round_stats)
    winner_counts = sdf['winner'].dropna().astype(int).value_counts()
    all_rank1 = load_all_rank1(exp_dir, mode, num_rounds)

    return {
        'winner':       winner_counts.idxmax() if len(winner_counts) else None,
        'winner_count': int(winner_counts.iloc[0]) if len(winner_counts) else 0,
        'mean_E':       float(sdf['entropy_E'].mean()),
        'mean_tax':     float(sdf['mean_tax'].mean()),
        'mean_fare':    float(sdf['mean_fare'].mean()),
        'mean_fee':     float(sdf['mean_fee'].mean()),
        'lever_e_tax':  compute_lever_entropy(all_rank1, policy_df, 'tax_percentage') if len(all_rank1) else np.nan,
        'lever_e_fare': compute_lever_entropy(all_rank1, policy_df, 'fare')           if len(all_rank1) else np.nan,
        'lever_e_fee':  compute_lever_entropy(all_rank1, policy_df, 'driver_fee')     if len(all_rank1) else np.nan,
    }


def fmt_mean_e(mean, e):
    return f'{mean:.3f} ({e:.2f})'

In [ ]:
# Table 3 — Chicago, 10 rounds
policy_df_chi = load_policy_params(POLICY_CSV_CHI)
table3_rows = []

for label, subdir in CHI_EXPERIMENTS.items():
    exp_dir = RESULTS_ROOT / subdir
    if not exp_dir.exists(): print(f'Skip: {subdir}'); continue
    mode = 'approval' if 'approval' in subdir else 'ranked'
    s = multi_round_summary(exp_dir, policy_df_chi, mode)
    if s: s['experiment'] = label; table3_rows.append(s)

if table3_rows:
    t3 = pd.DataFrame(table3_rows)
    t3_fmt = pd.DataFrame({
        'Experiment': t3['experiment'],
        'Winner':     t3['winner'],
        'Win Count':  t3['winner_count'],
        'Ē':          t3['mean_E'].apply('{:.3f}'.format),
        'tax (e_t)':  [fmt_mean_e(m,e) for m,e in zip(t3['mean_tax'],  t3['lever_e_tax'])],
        'fare (e_r)': [fmt_mean_e(m,e) for m,e in zip(t3['mean_fare'], t3['lever_e_fare'])],
        'fee (e_τ)':  [fmt_mean_e(m,e) for m,e in zip(t3['mean_fee'],  t3['lever_e_fee'])],
    })
    print('Table 3 — Multi-round summary (Chicago):')
    display(t3_fmt)
else:
    print('No data — run simulations first.')

In [ ]:
# Table 3 — Houston, 10 rounds
policy_df_hou = load_policy_params(POLICY_CSV_HOU)
table3_hou = []

for label, subdir in HOU_EXPERIMENTS.items():
    exp_dir = RESULTS_ROOT / subdir
    if not exp_dir.exists(): print(f'Skip: {subdir}'); continue
    mode = 'approval' if 'approval' in subdir else 'ranked'
    s = multi_round_summary(exp_dir, policy_df_hou, mode)
    if s: s['experiment'] = label; table3_hou.append(s)

if table3_hou:
    t3h = pd.DataFrame(table3_hou)
    t3h_fmt = pd.DataFrame({
        'Experiment': t3h['experiment'],
        'Winner':     t3h['winner'],
        'Win Count':  t3h['winner_count'],
        'Ē':          t3h['mean_E'].apply('{:.3f}'.format),
        'tax (e_t)':  [fmt_mean_e(m,e) for m,e in zip(t3h['mean_tax'],  t3h['lever_e_tax'])],
        'fare (e_r)': [fmt_mean_e(m,e) for m,e in zip(t3h['mean_fare'], t3h['lever_e_fare'])],
        'fee (e_τ)':  [fmt_mean_e(m,e) for m,e in zip(t3h['mean_fee'],  t3h['lever_e_fee'])],
    })
    print('Table 3 — Multi-round summary (Houston):')
    display(t3h_fmt)
else:
    print('No Houston data.')

## 7. Figure 6 / 7 — 3D Policy Space Bubble Charts

3D scatter in (tax × fare × driver_fee) policy space.  
One subplot per rank position; bubble size ∝ vote count across all rounds.

In [ ]:
def plot_3d_policy_votes(
    exp_dir: Path, policy_df: pd.DataFrame,
    num_rounds: int = NUM_ROUNDS, ranks: list = [1,2,3,4,5],
    title: str = '', save_path=None,
):
    """3D bubble chart in (tax × fare × fee) space, one subplot per rank position."""
    fig = plt.figure(figsize=(4.5 * len(ranks), 4.5))

    for idx, rp in enumerate(ranks):
        vote_counts = Counter()
        for i in range(1, num_rounds+1):
            csv_path = exp_dir / f'round_{i}' / 'combined_voting_results.csv'
            if not csv_path.exists(): continue
            df = pd.read_csv(csv_path)
            rc = f'rank{rp}'
            if rc in df.columns:
                for v in df[rc].dropna().astype(int): vote_counts[v] += 1

        ax = fig.add_subplot(1, len(ranks), idx+1, projection='3d')
        xs, ys, zs, sizes, colors = [], [], [], [], []
        for pid, cnt in vote_counts.items():
            if pid in policy_df.index:
                r = policy_df.loc[pid]
                xs.append(r['tax_percentage']); ys.append(r['fare'])
                zs.append(r['driver_fee']); sizes.append(cnt * 15); colors.append(cnt)

        if xs:
            sc = ax.scatter(xs, ys, zs, s=sizes, c=colors, cmap='YlOrRd',
                            alpha=0.85, edgecolors='grey', linewidth=0.3)
            plt.colorbar(sc, ax=ax, shrink=0.5, label='Votes')
        ax.set_xlabel('Tax (%)', fontsize=8, labelpad=5)
        ax.set_ylabel('Fare ($)', fontsize=8, labelpad=5)
        ax.set_zlabel('Fee ($)', fontsize=8, labelpad=5)
        ax.set_title(f'Rank {rp}', fontsize=11)
        ax.set_xticks([0.5,1.0,1.5]); ax.set_yticks([0.75,1.25,1.75]); ax.set_zticks([0,0.5,1])
        ax.tick_params(labelsize=7)

    plt.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Figure 6 — CHI-com GPT-4o ranked
exp_dir = RESULTS_ROOT / CHI_EXPERIMENTS['CHI-com GPT-4o (ranked)']
if exp_dir.exists():
    plot_3d_policy_votes(exp_dir, policy_df_chi, title='Figure 6 — CHI-com GPT-4o (ranked)',
                         save_path=FIG_DIR / 'fig6_3d_chi_gpt_ranked.png')
else:
    print(f'Not found: {exp_dir}')

In [ ]:
# Figure 7 — CHI-com Claude ranked
exp_dir = RESULTS_ROOT / CHI_EXPERIMENTS['CHI-com Claude (ranked)']
if exp_dir.exists():
    plot_3d_policy_votes(exp_dir, policy_df_chi, title='Figure 7 — CHI-com Claude (ranked)',
                         save_path=FIG_DIR / 'fig7_3d_chi_claude_ranked.png')
else:
    print(f'Not found: {exp_dir}')

In [ ]:
# Mean & Entropy heatmaps per lever × rank (paper Figure 6/7 insets)
def compute_mean_entropy_by_rank(
    exp_dir: Path, policy_df: pd.DataFrame, num_rounds: int = NUM_ROUNDS
) -> tuple:
    """
    Returns (mean_df, entropy_df): 3×5 DataFrames (levers × ranks).
    lever rows: tax_percentage, fare, driver_fee
    rank cols: rank1 … rank5
    """
    levers = ['tax_percentage', 'fare', 'driver_fee']
    ranks  = [f'rank{j}' for j in range(1, 6)]
    level_maps = {
        'tax_percentage': {0.5: 0.5, 1.0: 1.0, 1.5: 1.5},
        'fare':           {0.75: 0.75, 1.25: 1.25, 1.75: 1.75},
        'driver_fee':     {0.0: 0.0, 0.5: 0.5, 1.0: 1.0},
    }
    mean_acc  = np.zeros((3, 5))
    entr_acc  = np.zeros((3, 5))
    n_rounds  = 0

    for i in range(1, num_rounds+1):
        csv_path = exp_dir / f'round_{i}' / 'combined_voting_results.csv'
        if not csv_path.exists(): continue
        df = pd.read_csv(csv_path)
        for j, rc in enumerate(ranks):
            if rc not in df.columns: continue
            pids = df[rc].dropna().astype(int)
            params = policy_df.loc[pids]
            for li, lev in enumerate(levers):
                vals = params[lev].map(level_maps[lev])
                mean_acc[li, j] += vals.mean()
                cnt = vals.value_counts(normalize=True).sort_index()
                entr_acc[li, j] += float(scipy_entropy(cnt, base=2))
        n_rounds += 1

    if n_rounds == 0: return pd.DataFrame(), pd.DataFrame()
    labels = ['tax', 'fare', 'driver_fee']
    mean_df = pd.DataFrame(mean_acc / n_rounds, index=labels, columns=ranks)
    entr_df = pd.DataFrame(entr_acc / n_rounds, index=labels, columns=ranks)
    return mean_df, entr_df


def plot_mean_entropy_heatmaps(mean_df, entr_df, title='', save_path=None):
    if mean_df.empty: print('No data.'); return
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={'width_ratios': [1.2, 1]})

    sns.heatmap(mean_df, annot=True, fmt='.3f', cmap='coolwarm', center=1,
                linewidths=0.5, linecolor='gray', vmin=0.5, vmax=1.5,
                annot_kws={'size': 13},
                cbar_kws={'label': 'Mean', 'ticks': [0.5, 0.75, 1.0, 1.25, 1.5]}, ax=axes[0])
    axes[0].set_title('Mean lever value by rank', fontsize=12)
    axes[0].tick_params(labelsize=11)

    sns.heatmap(entr_df, annot=True, fmt='.2f', cmap='YlGnBu',
                linewidths=0.5, linecolor='gray', vmin=0, vmax=1.58,
                annot_kws={'size': 13},
                cbar_kws={'label': 'Entropy (bits)', 'ticks': [0, 0.5, 1.0, 1.5]}, ax=axes[1])
    axes[1].set_title('Lever entropy by rank', fontsize=12)
    axes[1].tick_params(labelsize=11)

    plt.suptitle(title, fontsize=12, y=1.02)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
for label, subdir in [
    ('CHI-com GPT-4o (ranked)',  'chicago_gpt_ranked'),
    ('CHI-know GPT-4o (ranked)', 'chicago_gpt_ranked_info'),
    ('CHI-com Claude (ranked)',  'chicago_claude_ranked'),
    ('CHI-know Claude (ranked)', 'chicago_claude_ranked_info'),
]:
    exp_dir = RESULTS_ROOT / subdir
    if not exp_dir.exists(): print(f'Skip: {subdir}'); continue
    mean_df, entr_df = compute_mean_entropy_by_rank(exp_dir, policy_df_chi)
    plot_mean_entropy_heatmaps(
        mean_df, entr_df, title=label,
        save_path=FIG_DIR / f'mean_entropy_heatmap_{subdir}.png',
    )

## 8. Figure 8 — VADER Sentiment Analysis

Apply VADER to the `decision_rationale` field in each community JSON response. Compare GPT-4o vs Claude.

In [ ]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER_OK = True
    print('vaderSentiment available.')
except ImportError:
    print('Install vaderSentiment: pip install vaderSentiment')
    VADER_OK = False


def extract_vader_scores(exp_dir: Path, num_rounds: int = NUM_ROUNDS) -> pd.DataFrame:
    """Load decision_rationale from all community JSON responses and score with VADER."""
    if not VADER_OK: return pd.DataFrame()
    analyzer = SentimentIntensityAnalyzer()
    records = []
    for i in range(1, num_rounds+1):
        resp_dir = exp_dir / f'round_{i}' / 'responses'
        if not resp_dir.exists(): continue
        for jf in sorted(resp_dir.glob('*.json')):
            try:
                d = json.loads(jf.read_text())
                rationale = d.get('thinking', {}).get('decision_rationale', '')
                if not rationale: continue
                sc = analyzer.polarity_scores(rationale)
                records.append({
                    'community_area': d.get('community_area', jf.stem),
                    'round': i,
                    'rationale': rationale,
                    'compound': sc['compound'],
                    'pos': sc['pos'], 'neu': sc['neu'], 'neg': sc['neg'],
                })
            except Exception: pass
    return pd.DataFrame(records)

In [ ]:
# Collect VADER scores for GPT-4o and Claude (Chicago ranked)
sentiment_dfs = {}
for model_label, exp_key in [('GPT-4o', 'CHI-com GPT-4o (ranked)'), ('Claude', 'CHI-com Claude (ranked)')]:
    exp_dir = RESULTS_ROOT / CHI_EXPERIMENTS.get(exp_key, '')
    if not exp_dir.exists(): print(f'Skip {exp_key}'); continue
    df_s = extract_vader_scores(exp_dir)
    if not df_s.empty:
        df_s['model'] = model_label
        sentiment_dfs[model_label] = df_s
        print(f'{model_label}: n={len(df_s)}, mean compound={df_s["compound"].mean():.3f}')

combined_sent = pd.concat(sentiment_dfs.values(), ignore_index=True) if sentiment_dfs else pd.DataFrame()

In [ ]:
# Figure 8 — Violin plot of compound sentiment
if not combined_sent.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    sns.violinplot(data=combined_sent, x='model', y='compound', ax=axes[0],
                   palette='Set2', inner='quartile')
    axes[0].axhline(0, color='grey', ls='--', alpha=0.6)
    axes[0].set_xlabel('Model', fontsize=12); axes[0].set_ylabel('VADER compound', fontsize=12)
    axes[0].set_title('Compound sentiment', fontsize=12)

    long_df = combined_sent.melt(id_vars=['model'], value_vars=['pos','neu','neg'],
                                  var_name='component', value_name='score')
    sns.boxplot(data=long_df, x='component', y='score', hue='model', ax=axes[1], palette='Set2')
    axes[1].set_xlabel('Sentiment component', fontsize=12)
    axes[1].set_ylabel('Score', fontsize=12)
    axes[1].set_title('Pos / Neu / Neg breakdown', fontsize=12)
    axes[1].legend(title='Model', fontsize=10)

    plt.suptitle('Figure 8 — VADER Sentiment of Decision Rationale', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig8_vader_sentiment.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('\nDescriptive statistics:')
    display(combined_sent.groupby('model')[['compound','pos','neu','neg']].describe().round(3))
else:
    print('No sentiment data — run simulations first.')

In [ ]:
# Figure 8 — Sentiment trend across rounds
if not combined_sent.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    for model, grp in combined_sent.groupby('model'):
        rm = grp.groupby('round')['compound']
        ax.plot(rm.mean().index, rm.mean().values, marker='o', label=model)
        ax.fill_between(rm.mean().index, rm.quantile(0.25).values, rm.quantile(0.75).values, alpha=0.15)
    ax.axhline(0, color='grey', ls='--', alpha=0.5)
    ax.set_xlabel('Round', fontsize=12); ax.set_ylabel('Mean compound', fontsize=12)
    ax.set_title('VADER sentiment across rounds', fontsize=12); ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig8_vader_by_round.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Section 4.4 — OLS Regression

Regress community Borda-weighted mean voted lever values (tax, fare, fee) on demographic characteristics from `data/CA_info/`.

### 9.1 Build Community Demographic Table

In [ ]:
def sanitize_name(name: str) -> str:
    name = name.replace('/', '&')
    name = re.sub(r"[^\w\s&-]", '', name)
    return name.replace(' ', '_').lower()


def load_ca_info(ca_info_dir: Path) -> pd.DataFrame:
    """Load all CA_info JSON files into a flat DataFrame."""
    records = []
    for jf in sorted(ca_info_dir.glob('*.json')):
        try:
            d = json.loads(jf.read_text(encoding='utf-8'))
            row = {'community_file': jf.stem}
            for cat, vals in d.items():
                for k, v in vals.items():
                    row[f'{cat}: {k}'] = v
            records.append(row)
        except Exception as e:
            print(f'Error {jf.name}: {e}')
    return pd.DataFrame(records)


ca_raw = load_ca_info(CA_INFO_DIR)
print(f'Loaded {len(ca_raw)} communities from CA_info.')

In [ ]:
# Normalize and rename columns
ca_demo = ca_raw.rename(columns={
    'Population: Total Population':             'pop_total',
    'Population: Total Households':             'pop_households',
    'Population: Average Household Size':       'household_size',
    'Race: White (Non-Hispanic)':               'race_white',
    'Race: Hispanic or Latino (Any Race)':      'race_hispanic',
    'Race: Black (Non-Hispanic)':               'race_black',
    'Race: Asian (Non-Hispanic)':               'race_asian',
    'Race: Other/Multiple Races (Non-Hispanic)':'race_other',
    'Car Ownership: No Vehicle':                'car_no',
    'Car Ownership: 1 Vehicle':                 'car_1',
    'Car Ownership: 2 Vehicles':                'car_2',
    'Car Ownership: 3+ Vehicles':               'car_3plus',
    'Travel Mode: Work at Home':                'tvl_wfh',
    'Travel Mode: Drive Alone':                 'tvl_drive',
    'Travel Mode: Carpool':                     'tvl_carpool',
    'Travel Mode: Transit':                     'tvl_transit',
    'Travel Mode: Walk or Bike':                'tvl_walk_bike',
    'Travel Mode: Other':                       'tvl_other',
    'Income: Less than $25,000':                'income_lt25k',
    'Income: $25,000 to $49,999':               'income_25_49k',
    'Income: $50,000 to $74,999':               'income_50_74k',
    'Income: $75,000 to $99,999':               'income_75_99k',
    'Income: $100,000 to $149,999':             'income_100_149k',
    'Income: $150,000 and Over':                'income_150k_plus',
    'Income: Median Income':                    'median_income',
    'Income: Per Capita Income':                'per_capita_income',
})

# Percentage columns → fraction
pct_cols = [c for c in ca_demo.columns if any(c.startswith(p) for p in
    ['race_','car_','tvl_','income_lt','income_25','income_50','income_75','income_100','income_150'])]
ca_demo[pct_cols] = ca_demo[pct_cols] / 100.0
ca_demo['pct_non_white']  = 1.0 - ca_demo.get('race_white', 0)
ca_demo['pct_no_vehicle'] = ca_demo.get('car_no', 0)

print(f'Demographic table: {ca_demo.shape}')
ca_demo.head(3)

### 9.2 Borda-Weighted Mean Voted Policy Values per Community

In [ ]:
def compute_borda_means(exp_dir: Path, policy_df: pd.DataFrame,
                         num_rounds: int = NUM_ROUNDS, weights=(5,4,3,2,1)) -> pd.DataFrame:
    """
    Per-community Borda-weighted average of tax, fare, fee across 10 rounds.
    Borda weights: rank1=5, rank2=4, ..., rank5=1.
    """
    comm_data = {}   # {ca: {w_tax, w_fare, w_fee, w_total}}
    for i in range(1, num_rounds+1):
        csv_path = exp_dir / f'round_{i}' / 'combined_voting_results.csv'
        if not csv_path.exists(): continue
        df = pd.read_csv(csv_path)
        rank_cols = [f'rank{j}' for j in range(1, len(weights)+1) if f'rank{j}' in df.columns]
        for _, row in df.iterrows():
            ca = row['community_area']
            if ca not in comm_data:
                comm_data[ca] = dict(w_tax=0.0, w_fare=0.0, w_fee=0.0, w_total=0.0)
            for w, rc in zip(weights, rank_cols):
                val = row.get(rc)
                if pd.isna(val): continue
                pid = int(val)
                if pid not in policy_df.index: continue
                p = policy_df.loc[pid]
                comm_data[ca]['w_tax']   += w * p['tax_percentage']
                comm_data[ca]['w_fare']  += w * p['fare']
                comm_data[ca]['w_fee']   += w * p['driver_fee']
                comm_data[ca]['w_total'] += w

    rows = []
    for ca, d in comm_data.items():
        if d['w_total'] > 0:
            rows.append({
                'community_area': ca,
                'file_stem':      sanitize_name(ca),
                'borda_tax':      d['w_tax']  / d['w_total'],
                'borda_fare':     d['w_fare'] / d['w_total'],
                'borda_fee':      d['w_fee']  / d['w_total'],
            })
    return pd.DataFrame(rows)


print('compute_borda_means defined.')

In [ ]:
# Compute Borda means and merge with demographics
merged_dfs = {}
for label, subdir in [
    ('GPT-4o',       'chicago_gpt_ranked'),
    ('GPT-4o (info)','chicago_gpt_ranked_info'),
    ('Claude',       'chicago_claude_ranked'),
    ('Claude (info)','chicago_claude_ranked_info'),
]:
    exp_dir = RESULTS_ROOT / subdir
    if not exp_dir.exists(): print(f'Skip: {subdir}'); continue
    bm = compute_borda_means(exp_dir, policy_df_chi)
    if bm.empty: continue
    merged = pd.merge(bm, ca_demo, left_on='file_stem', right_on='community_file', how='inner')
    merged_dfs[label] = merged
    print(f'{label}: {len(merged)} communities merged.')

### 9.3 OLS Regression

In [ ]:
try:
    import statsmodels.api as sm
    from sklearn.preprocessing import StandardScaler
    SM_OK = True
except ImportError:
    print('pip install statsmodels scikit-learn')
    SM_OK = False


def run_ols(data: pd.DataFrame, x_cols: list, y_col: str, standardize: bool = True):
    """Standardized OLS regression. Returns fitted model or None."""
    if not SM_OK: return None
    sub = data[x_cols + [y_col]].dropna()
    if sub.empty: return None
    X = sub[x_cols].astype(float)
    y = sub[y_col].astype(float)
    if standardize:
        X = pd.DataFrame(StandardScaler().fit_transform(X), columns=x_cols, index=X.index)
    return sm.OLS(y, sm.add_constant(X)).fit()


# Key predictors matching the paper
X_COLS = ['household_size', 'pct_non_white', 'car_no', 'tvl_transit', 'income_lt25k', 'income_150k_plus']

print('OLS functions defined.')

In [ ]:
# Run OLS for each experiment × each lever
reg_results = {}

for label, merged in merged_dfs.items():
    reg_results[label] = {}
    x_avail = [c for c in X_COLS if c in merged.columns]
    print(f'\n{'='*60}\n{label}\n{'='*60}')
    for lever_name, y_col in [('Tax rate', 'borda_tax'), ('Fare', 'borda_fare'), ('Driver fee', 'borda_fee')]:
        model = run_ols(merged, x_avail, y_col)
        reg_results[label][lever_name] = model
        if model:
            print(f'\n--- Y = {lever_name} | R²={model.rsquared:.3f}, adj-R²={model.rsquared_adj:.3f} ---')
            tbl = model.summary2().tables[1][['Coef.','Std.Err.','t','P>|t|']].drop('const', errors='ignore')
            display(tbl.round(4))

In [ ]:
# Coefficient comparison plot: GPT-4o vs Claude
def plot_coef_comparison(reg_results: dict, lever: str, x_cols: list, figsize=(9, 5)):
    if not SM_OK: return
    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(x_cols))
    bw = 0.35
    colors = ['steelblue', 'tomato', 'seagreen', 'orchid']

    for i, (label, lever_models) in enumerate(reg_results.items()):
        model = lever_models.get(lever)
        if not model: continue
        coef = model.params.drop('const', errors='ignore')
        ci   = model.conf_int().drop('const', errors='ignore')
        vals  = np.array([coef.get(c, np.nan) for c in x_cols])
        lo    = np.array([ci.loc[c, 0] if c in ci.index else np.nan for c in x_cols])
        hi    = np.array([ci.loc[c, 1] if c in ci.index else np.nan for c in x_cols])

        offset = (i - len(reg_results)/2 + 0.5) * bw
        ax.bar(x + offset, vals, bw, yerr=[vals-lo, hi-vals],
               capsize=3, color=colors[i % len(colors)], alpha=0.75,
               label=label, error_kw={'linewidth': 1.2})

    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xticks(x); ax.set_xticklabels(x_cols, rotation=35, ha='right', fontsize=10)
    ax.set_ylabel('Standardized coefficient', fontsize=11)
    ax.set_title(f'OLS coefficients — Y = {lever}', fontsize=12)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'reg_coef_{lever.replace(" ","_")}.png', dpi=300, bbox_inches='tight')
    plt.show()


if reg_results and SM_OK:
    all_merged = list(merged_dfs.values())
    if all_merged:
        x_avail = [c for c in X_COLS if c in all_merged[0].columns]
        for lever in ['Tax rate', 'Fare', 'Driver fee']:
            plot_coef_comparison(reg_results, lever, x_avail)

## 10. Section 4.5 — Houston vs Chicago Comparison

Compare entropy, winning policy, and mean lever values between cities.

In [ ]:
# Build combined comparison table
compare_rows = []
configs = [
    ('Chicago', 'GPT-4o Ranked',       'chicago_gpt_ranked',        policy_df_chi, 'ranked'),
    ('Houston', 'GPT-4o Ranked',       'houston_gpt_ranked',        policy_df_hou, 'ranked'),
    ('Chicago', 'Claude Ranked',       'chicago_claude_ranked',     policy_df_chi, 'ranked'),
    ('Houston', 'Claude Ranked',       'houston_claude_ranked',     policy_df_hou, 'ranked'),
    ('Chicago', 'GPT-4o Approval-All', 'chicago_gpt_approval-all',  policy_df_chi, 'approval-all'),
    ('Houston', 'GPT-4o Approval-All', 'houston_gpt_approval-all',  policy_df_hou, 'approval-all'),
    ('Chicago', 'Claude Approval-All', 'chicago_claude_approval-all',policy_df_chi,'approval-all'),
    ('Houston', 'Claude Approval-All', 'houston_claude_approval-all',policy_df_hou,'approval-all'),
]

for city, exp_label, subdir, pdf, mode in configs:
    exp_dir = RESULTS_ROOT / subdir
    if not exp_dir.exists(): print(f'Skip: {subdir}'); continue
    s = multi_round_summary(exp_dir, pdf, mode)
    if s:
        s['city'] = city; s['experiment'] = exp_label
        compare_rows.append(s)

if compare_rows:
    cdf = pd.DataFrame(compare_rows)
    cdf_fmt = pd.DataFrame({
        'City':        cdf['city'],
        'Experiment':  cdf['experiment'],
        'Winner':      cdf['winner'],
        'Win Count':   cdf['winner_count'],
        'Ē':           cdf['mean_E'].apply('{:.3f}'.format),
        'tax (e_t)':   [fmt_mean_e(m,e) for m,e in zip(cdf['mean_tax'],  cdf['lever_e_tax'])],
        'fare (e_r)':  [fmt_mean_e(m,e) for m,e in zip(cdf['mean_fare'], cdf['lever_e_fare'])],
        'fee (e_τ)':   [fmt_mean_e(m,e) for m,e in zip(cdf['mean_fee'],  cdf['lever_e_fee'])],
    })
    print('Chicago vs Houston full comparison:')
    display(cdf_fmt)
else:
    print('No data — run simulations first.')

In [ ]:
# Grouped bar chart: entropy by city × experiment
if compare_rows:
    cdf = pd.DataFrame(compare_rows)
    fig, ax = plt.subplots(figsize=(10, 4))

    experiments = cdf['experiment'].unique()
    cities = cdf['city'].unique()
    x = np.arange(len(experiments))
    bw = 0.35

    for i, city in enumerate(cities):
        sub = cdf[cdf['city'] == city]
        e_vals = [sub[sub['experiment']==g]['mean_E'].values[0] if len(sub[sub['experiment']==g]) else np.nan
                  for g in experiments]
        ax.bar(x + (i-0.5)*bw, e_vals, bw, label=city, alpha=0.8)

    ax.set_xticks(x); ax.set_xticklabels(experiments, rotation=20, ha='right', fontsize=10)
    ax.set_ylabel('Mean Entropy Ē (bits)', fontsize=12)
    ax.set_title('Policy entropy — Chicago vs Houston', fontsize=12)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig_city_entropy_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Side-by-side 3D rank-1 plots: Chicago vs Houston (GPT-4o ranked)
fig = plt.figure(figsize=(12, 5))
for plot_idx, (city, subdir, pdf) in enumerate([
    ('Chicago', 'chicago_gpt_ranked', policy_df_chi),
    ('Houston', 'houston_gpt_ranked', policy_df_hou),
], start=1):
    ax = fig.add_subplot(1, 2, plot_idx, projection='3d')
    exp_dir = RESULTS_ROOT / subdir
    vc = Counter()
    if exp_dir.exists():
        for i in range(1, NUM_ROUNDS+1):
            cp = exp_dir / f'round_{i}' / 'combined_voting_results.csv'
            if cp.exists():
                df = pd.read_csv(cp)
                if 'rank1' in df.columns:
                    for v in df['rank1'].dropna().astype(int): vc[v] += 1
    xs,ys,zs,ss,cs = [],[],[],[],[]
    for pid,cnt in vc.items():
        if pid in pdf.index:
            r = pdf.loc[pid]
            xs.append(r['tax_percentage']); ys.append(r['fare'])
            zs.append(r['driver_fee']); ss.append(cnt*15); cs.append(cnt)
    if xs:
        sc = ax.scatter(xs,ys,zs,s=ss,c=cs,cmap='YlOrRd',alpha=0.85,edgecolors='grey',linewidth=0.4)
        plt.colorbar(sc, ax=ax, shrink=0.5, label='Rank-1 votes')
    ax.set_xlabel('Tax (%)',fontsize=9); ax.set_ylabel('Fare ($)',fontsize=9)
    ax.set_zlabel('Fee ($)',fontsize=9); ax.set_title(f'{city} — GPT-4o Ranked',fontsize=11)
    ax.set_xticks([0.5,1.0,1.5]); ax.set_yticks([0.75,1.25,1.75]); ax.set_zticks([0,0.5,1])

plt.suptitle('Rank-1 votes: Chicago vs Houston', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_city_comparison_3d.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

| Section | Output |
|---------|--------|
| Fig 4 | Policy utility scatter plots ($U_k$ vs $u_k$, $\gamma_k$, $G_k$) for Chicago and Houston |
| Fig 5 | Vote count bar charts comparing ranked / approval-5 / approval-all (GPT-4o) |
| Table 2 | Single-round winner, lever values, mean values, entropy $E$ |
| Table 3 | Multi-round summary: winner counts, $\bar{E}$, mean lever values with lever entropy |
| Fig 6/7 | 3D policy space bubble charts by rank position (GPT-4o and Claude) |
| Heatmaps | Mean and entropy of lever values by rank (Fig 6/7 insets) |
| Fig 8 | VADER sentiment of decision rationale — violin and box plots |
| Sec 4.4 | OLS regression tables and coefficient comparison plots |
| Sec 4.5 | Chicago vs Houston entropy comparison and 3D rank-1 plots |

All figures are saved to `./figures/`.